<a href="https://colab.research.google.com/github/Misbah361/60-days-AI-challenge/blob/main/Day8_Text_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 8: Text Preprocessing Pipeline
Cleaning noisy real-world text into structured, machine-readable tokens using **Python, NLTK and Regex**.

Pipeline: Tokenisation → Stopword removal → Punctuation stripping → Stemming vs Lemmatisation → Before/After comparison

In [1]:
import re
import nltk
for pkg in ['punkt','punkt_tab','stopwords','wordnet','omw-1.4']:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
print('Setup complete. Stopwords loaded:', len(stop_words))

Setup complete. Stopwords loaded: 198


## Step 0: Load three raw text samples
Social media post, news article, customer review. All contain punctuation, emojis, stopwords and inconsistent casing.

In [2]:
samples = {
    'Social Media': (
        "OMG just watched the NEW Marvel movie!!! 😍🔥 It was SOOO good, "
        "I can't even... the ending literally made me cry 😭😭 #MustWatch @MarvelStudios "
        "Everyone NEEDS to go see it this weekend!!!"
    ),
    'News Article': (
        "The Government announced on Tuesday that it will be investing $2.5 billion in renewable energy projects. "
        "Officials said the new policy is expected to create thousands of jobs across the country. "
        "Critics, however, argue that the plan does not go far enough to tackle climate change!"
    ),
    'Customer Review': (
        "I bought this laptop two weeks ago and I am REALLY disappointed... The battery barely lasts 3 hours, "
        "the screen is flickering, and customer support wasn't helpful at all 😡. "
        "Would NOT recommend it to anyone!!! Total waste of money."
    ),
}
for name, text in samples.items():
    print(f'--- {name} ---')
    print(text, '\n')

--- Social Media ---
OMG just watched the NEW Marvel movie!!! 😍🔥 It was SOOO good, I can't even... the ending literally made me cry 😭😭 #MustWatch @MarvelStudios Everyone NEEDS to go see it this weekend!!! 

--- News Article ---
The Government announced on Tuesday that it will be investing $2.5 billion in renewable energy projects. Officials said the new policy is expected to create thousands of jobs across the country. Critics, however, argue that the plan does not go far enough to tackle climate change! 

--- Customer Review ---
I bought this laptop two weeks ago and I am REALLY disappointed... The battery barely lasts 3 hours, the screen is flickering, and customer support wasn't helpful at all 😡. Would NOT recommend it to anyone!!! Total waste of money. 



## Step 1: Tokenisation (word-level and sentence-level)
Tokenisation splits raw text into units a model can work with.

In [3]:
sent_tokens, word_tokens = {}, {}
for name, text in samples.items():
    sent_tokens[name] = sent_tokenize(text)
    word_tokens[name] = word_tokenize(text)
    print(f'=== {name} ===')
    print('SENTENCES:')
    for i, s in enumerate(sent_tokens[name], 1):
        print(f'  {i}. {s}')
    print('WORDS:', word_tokens[name])
    print(f'Counts -> sentences: {len(sent_tokens[name])}, word tokens: {len(word_tokens[name])}\n')

=== Social Media ===
SENTENCES:
  1. OMG just watched the NEW Marvel movie!!!
  2. 😍🔥 It was SOOO good, I can't even... the ending literally made me cry 😭😭 #MustWatch @MarvelStudios Everyone NEEDS to go see it this weekend!!
  3. !
WORDS: ['OMG', 'just', 'watched', 'the', 'NEW', 'Marvel', 'movie', '!', '!', '!', '😍🔥', 'It', 'was', 'SOOO', 'good', ',', 'I', 'ca', "n't", 'even', '...', 'the', 'ending', 'literally', 'made', 'me', 'cry', '😭😭', '#', 'MustWatch', '@', 'MarvelStudios', 'Everyone', 'NEEDS', 'to', 'go', 'see', 'it', 'this', 'weekend', '!', '!', '!']
Counts -> sentences: 3, word tokens: 43

=== News Article ===
SENTENCES:
  1. The Government announced on Tuesday that it will be investing $2.5 billion in renewable energy projects.
  2. Officials said the new policy is expected to create thousands of jobs across the country.
  3. Critics, however, argue that the plan does not go far enough to tackle climate change!
WORDS: ['The', 'Government', 'announced', 'on', 'Tuesday', 'that',

## Step 2: Stopword removal (NLTK corpus) + vocabulary size comparison
Tokens are lowercased first so that 'The' and 'the' are treated identically.

In [4]:
no_stop = {}
print(f"{'Sample':<18}{'Vocab before':>14}{'Vocab after':>14}{'Reduction':>12}")
for name in samples:
    lowered = [t.lower() for t in word_tokens[name]]
    no_stop[name] = [t for t in lowered if t not in stop_words]
    before, after = len(set(lowered)), len(set(no_stop[name]))
    print(f'{name:<18}{before:>14}{after:>14}{(before-after)/before*100:>11.1f}%')
print()
for name in samples:
    print(f'{name}\n  BEFORE: {[t.lower() for t in word_tokens[name]]}\n  AFTER : {no_stop[name]}\n')

Sample              Vocab before   Vocab after   Reduction
Social Media                  36            28       22.2%
News Article                  45            33       26.7%
Customer Review               42            29       31.0%

Social Media
  BEFORE: ['omg', 'just', 'watched', 'the', 'new', 'marvel', 'movie', '!', '!', '!', '😍🔥', 'it', 'was', 'sooo', 'good', ',', 'i', 'ca', "n't", 'even', '...', 'the', 'ending', 'literally', 'made', 'me', 'cry', '😭😭', '#', 'mustwatch', '@', 'marvelstudios', 'everyone', 'needs', 'to', 'go', 'see', 'it', 'this', 'weekend', '!', '!', '!']
  AFTER : ['omg', 'watched', 'new', 'marvel', 'movie', '!', '!', '!', '😍🔥', 'sooo', 'good', ',', 'ca', "n't", 'even', '...', 'ending', 'literally', 'made', 'cry', '😭😭', '#', 'mustwatch', '@', 'marvelstudios', 'everyone', 'needs', 'go', 'see', 'weekend', '!', '!', '!']

News Article
  BEFORE: ['the', 'government', 'announced', 'on', 'tuesday', 'that', 'it', 'will', 'be', 'investing', '$', '2.5', 'billion', 'in', 

## Step 3: Strip punctuation (and emojis/symbols) with regex

In [5]:
def strip_punct(tokens):
    # WHY THIS STEP IS NECESSARY:
    # Punctuation and emojis carry almost no meaning for bag-of-words/TF-IDF models, but they
    # inflate the vocabulary: 'good', 'good!' and 'good!!!' would become three different features
    # even though they mean the same thing. Removing them merges these duplicates, shrinks the
    # feature space and makes counts/embeddings more reliable downstream.
    cleaned = [re.sub(r'[^a-z0-9\s]', '', t) for t in tokens]   # keep only letters, digits, spaces
    return [t for t in cleaned if t.strip()]                      # drop tokens that became empty

no_punct = {}
for name in samples:
    no_punct[name] = strip_punct(no_stop[name])
    print(f'=== {name} ===')
    print('BEFORE:', no_stop[name])
    print('AFTER :', no_punct[name], '\n')

=== Social Media ===
BEFORE: ['omg', 'watched', 'new', 'marvel', 'movie', '!', '!', '!', '😍🔥', 'sooo', 'good', ',', 'ca', "n't", 'even', '...', 'ending', 'literally', 'made', 'cry', '😭😭', '#', 'mustwatch', '@', 'marvelstudios', 'everyone', 'needs', 'go', 'see', 'weekend', '!', '!', '!']
AFTER : ['omg', 'watched', 'new', 'marvel', 'movie', 'sooo', 'good', 'ca', 'nt', 'even', 'ending', 'literally', 'made', 'cry', 'mustwatch', 'marvelstudios', 'everyone', 'needs', 'go', 'see', 'weekend'] 

=== News Article ===
BEFORE: ['government', 'announced', 'tuesday', 'investing', '$', '2.5', 'billion', 'renewable', 'energy', 'projects', '.', 'officials', 'said', 'new', 'policy', 'expected', 'create', 'thousands', 'jobs', 'across', 'country', '.', 'critics', ',', 'however', ',', 'argue', 'plan', 'go', 'far', 'enough', 'tackle', 'climate', 'change', '!']
AFTER : ['government', 'announced', 'tuesday', 'investing', '25', 'billion', 'renewable', 'energy', 'projects', 'officials', 'said', 'new', 'policy',

## Step 4: Stemming (PorterStemmer) vs Lemmatisation (WordNetLemmatizer), side by side

In [6]:
import pandas as pd
stemmed, lemmatized = {}, {}
for name in samples:
    toks = no_punct[name]
    stemmed[name] = [stemmer.stem(t) for t in toks]
    lemmatized[name] = [lemmatizer.lemmatize(t) for t in toks]
    df = pd.DataFrame({'original': toks, 'stem (Porter)': stemmed[name], 'lemma (WordNet)': lemmatized[name]})
    print(f'=== {name} ===')
    print(df.to_string(index=False), '\n')

# Words where the two methods differ
for name in samples:
    diff = [(o, s, l) for o, s, l in zip(no_punct[name], stemmed[name], lemmatized[name]) if s != l]
    print(name, 'differences (original, stem, lemma):', diff)

=== Social Media ===
     original stem (Porter) lemma (WordNet)
          omg           omg             omg
      watched         watch         watched
          new           new             new
       marvel        marvel          marvel
        movie          movi           movie
         sooo          sooo            sooo
         good          good            good
           ca            ca              ca
           nt            nt              nt
         even          even            even
       ending           end          ending
    literally         liter       literally
         made          made            made
          cry           cri             cry
    mustwatch     mustwatch       mustwatch
marvelstudios  marvelstudio   marvelstudios
     everyone       everyon        everyone
        needs          need            need
           go            go              go
          see           see             see
      weekend       weekend         weekend 

=== News 

## Step 5: Before-and-after comparison for all five steps across all three samples

In [7]:
for name, text in samples.items():
    low = [t.lower() for t in word_tokens[name]]
    stages = [
        ('1a. Sentence tokenisation', text, sent_tokens[name]),
        ('1b. Word tokenisation', text, word_tokens[name]),
        ('2. Stopword removal', low, no_stop[name]),
        ('3. Punctuation stripping', no_stop[name], no_punct[name]),
        ('4a. Stemming (Porter)', no_punct[name], stemmed[name]),
        ('4b. Lemmatisation (WordNet)', no_punct[name], lemmatized[name]),
    ]
    print('#' * 80); print(f'SAMPLE: {name}'); print('#' * 80)
    for title, before, after in stages:
        print(f'\n[{title}]')
        print('  BEFORE:', before)
        print('  AFTER :', after)
    print(f"\nFinal vocabulary: {len(set(word_tokens[name]))} raw -> {len(set(lemmatized[name]))} cleaned\n")

################################################################################
SAMPLE: Social Media
################################################################################

[1a. Sentence tokenisation]
  BEFORE: OMG just watched the NEW Marvel movie!!! 😍🔥 It was SOOO good, I can't even... the ending literally made me cry 😭😭 #MustWatch @MarvelStudios Everyone NEEDS to go see it this weekend!!!
  AFTER : ['OMG just watched the NEW Marvel movie!!!', "😍🔥 It was SOOO good, I can't even... the ending literally made me cry 😭😭 #MustWatch @MarvelStudios Everyone NEEDS to go see it this weekend!!", '!']

[1b. Word tokenisation]
  BEFORE: OMG just watched the NEW Marvel movie!!! 😍🔥 It was SOOO good, I can't even... the ending literally made me cry 😭😭 #MustWatch @MarvelStudios Everyone NEEDS to go see it this weekend!!!
  AFTER : ['OMG', 'just', 'watched', 'the', 'NEW', 'Marvel', 'movie', '!', '!', '!', '😍🔥', 'It', 'was', 'SOOO', 'good', ',', 'I', 'ca', "n't", 'even', '...', 'the', 'endi

## Step 6: Why each preprocessing step matters

| Step | Why it matters | What breaks if you skip it |
|---|---|---|
| **Tokenisation** | Turns a raw string into discrete units (words/sentences) that models can count or embed. | Nothing downstream works; the model sees one long string. Splitting on spaces alone also glues punctuation to words ("movie!!!"). |
| **Stopword removal** | Drops high-frequency words ("the", "is", "and") that add little meaning, so signal words stand out. | Common words dominate counts, TF-IDF and similarity scores; vocabulary and compute grow; search and classification get noisier. |
| **Punctuation stripping** | Merges variants like "good", "good!" and "good!!!" into one feature and removes emojis/symbols. | Vocabulary explodes with near-duplicate tokens, features get sparse, and the model learns spurious patterns. |
| **Stemming** | Fast rule-based reduction of words to a root ("running" → "run", but also "studies" → "studi"). | Related forms ("recommend", "recommended") count as separate words, diluting frequency. Stems may be non-words though. |
| **Lemmatisation** | Dictionary-based reduction to a real base word ("better" → "good" with POS). More accurate, slower. | Same word-form fragmentation as above, and without it you cannot get clean, readable normalised vocabulary. |

**Stemming vs lemmatisation:** stemming is faster but crude; lemmatisation gives valid words and is better when meaning matters (sentiment, search, chatbots). Also, lowercasing is applied before stopword removal; skipping it makes "The" and "the" separate tokens and stopword matching fails.

**Takeaway:** Preprocessing is the foundation of every NLP pipeline. Garbage in, garbage out: a clean token stream gives smaller vocabularies, stronger signal and better models.